# 03 - Build Silver Layer

Este notebook constrói a camada Silver a partir da tabela Raw.

## Objetivos

- Ler a tabela Raw.
- Aplicar regras mínimas de qualidade.
- Padronizar os tipos das colunas.
- Garantir as colunas obrigatórias do case.
- Criar colunas auxiliares para análise por data, mês e hora.
- Persistir a tabela Silver no Unity Catalog.

A Silver representa a camada limpa e preparada para consumo analítico.

---

## 00. Criação da camada prata

#### 01. Libs Usadas

In [1]:
import os
import sys
import importlib

#### 02. Import do projeto

In [2]:
current_path = os.getcwd()

if current_path.endswith("/notebooks"):
    PROJECT_ROOT = os.path.abspath("..")
else:
    PROJECT_ROOT = current_path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for module_name in ["src.build_silver"]:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

import src.build_silver as silver

print("Raw table:", silver.RAW_TABLE_NAME)
print("Silver table:", silver.SILVER_TABLE_NAME)

from src.build_silver import create_silver

ModuleNotFoundError: No module named 'src'

### 03. Criar tabela Silver

In [ ]:
silver_df = create_silver(
    spark=spark,
    mode="overwrite",
)

print(f"Silver records: {silver_df.count()}")

display(silver_df.limit(10))

---

## 01. Validações da Silver

#### 01. Contagem por mês e Schema

In [ ]:
display(
    spark.table("workspace.ifood_case.silver_yellow_taxi_trips")
    .groupBy("pickup_year", "pickup_month")
    .count()
    .orderBy("pickup_year", "pickup_month")
)

spark.table("workspace.ifood_case.silver_yellow_taxi_trips").printSchema()

#### 02. Validar Colunas obrigatórias

In [ ]:
required_columns = {
    "VendorID",
    "passenger_count",
    "total_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
}

silver_columns = set(
    spark.table("workspace.ifood_case.silver_yellow_taxi_trips").columns
)

missing_columns = required_columns - silver_columns

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available in Silver table.")

#### 03. Validar nulos nas colunas obrigatórias

In [ ]:
from pyspark.sql import functions as F

silver_table = spark.table("workspace.ifood_case.silver_yellow_taxi_trips")

display(
    silver_table.select(
        *[
            F.count(
                F.when(F.col(column).isNull(), column)
            ).alias(f"{column}_nulls")
            for column in [
                "VendorID",
                "passenger_count",
                "total_amount",
                "tpep_pickup_datetime",
                "tpep_dropoff_datetime",
            ]
        ]
    )
)

#### 04. Validar qualidade básica

In [ ]:
display(
    silver_table.select(
        F.count("*").alias("total_records"),
        F.sum(F.when(F.col("passenger_count") <= 0, 1).otherwise(0)).alias("invalid_passenger_count"),
        F.sum(F.when(F.col("total_amount") < 0, 1).otherwise(0)).alias("negative_total_amount"),
        F.sum(
            F.when(
                F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"),
                1,
            ).otherwise(0)
        ).alias("invalid_trip_time"),
        F.min("tpep_pickup_datetime").alias("min_pickup_datetime"),
        F.max("tpep_pickup_datetime").alias("max_pickup_datetime"),
    )
)

#### 05. Confirmar tabela no catálogo

In [ ]:
display(
    spark.sql("SHOW TABLES IN workspace.ifood_case")
)